# Representative Independent Analytical Project: Healthcare Billing & Operational Data Analysis
**Author:** Christine Nabisswa  
**Role:** Healthcare Operations & Data Analytics
**Toolstack:** Python, SQLite (SQL), Pandas, OpenPyXL (Excel Data Modeling)
**Document Purpose:** Demonstration of SQL relational database querying, Common Table Expressions (CTEs), dynamic data discretization, and cross-tabulation analysis applied to healthcare billing and reimbursement datasets.
---

In [ ]:
import pandas as pd
import sqlite3
import numpy as np

# Load the uploaded file from Colab storage
df_billing = pd.read_csv("healthcare_dataset.csv")

# Clean column names for SQL compatibility
df_billing.columns = df_billing.columns.str.strip().str.replace(' ', '_').str.lower()
# Format dates and billing amounts
df_billing['date_of_admission'] = pd.to_datetime(df_billing['date_of_admission'])
df_billing['discharge_date'] = pd.to_datetime(df_billing['discharge_date'])
df_billing['billing_amount'] = pd.to_numeric(df_billing['billing_amount'], errors='coerce').round(2)

# Calculate length of stay (days)
df_billing['length_of_stay'] = (df_billing['discharge_date'] - df_billing['date_of_admission']).dt.days

# Load into SQLite database in Colab
conn = sqlite3.connect(':memory:')
df_billing.to_sql('billing_claims', conn, index=False, if_exists='replace')

print("Project Author: Christine Nabisswa")
print(f"Successfully loaded {len(df_billing):,} billing claims into SQL database for analysis.")
df_billing[['name', 'medical_condition', 'insurance_provider', 'billing_amount', 'admission_type', 'length_of_stay']].head(5)

Project Author: Christine Nabisswa
Successfully loaded 55,500 billing claims into SQL database for analysis.


,name,medical_condition,insurance_provider,billing_amount,admission_type,length_of_stay
0,Bobby JacksOn,Cancer,Blue Cross,"18,856.28",Urgent,2
1,LesLie TErRy,Obesity,Medicare,"33,643.33",Emergency,6
2,DaNnY sMitH,Obesity,Aetna,"27,955.10",Emergency,15
3,andrEw waTtS,Diabetes,Medicare,"37,909.78",Elective,30
4,adrIENNE bEll,Cancer,Aetna,"14,238.32",Urgent,20


In [ ]:
# SQL Query: Revenue share, average stays, and payer ranking
pd.options.display.float_format = '{:,.2f}'.format

sql_query = """
WITH Payer_Summary AS (
    SELECT
        insurance_provider,
        COUNT(*) AS total_claims,
        ROUND(SUM(billing_amount), 2) AS total_billed,
        ROUND(AVG(billing_amount), 2) AS avg_claim_amount,
        ROUND(AVG(length_of_stay), 1) AS avg_days_stay
    FROM billing_claims
    GROUP BY insurance_provider
)
SELECT
    insurance_provider,
    total_claims,
    total_billed,
    avg_claim_amount,
    avg_days_stay,
    RANK() OVER (ORDER BY total_billed DESC) AS revenue_rank
FROM Payer_Summary
ORDER BY revenue_rank;
"""

df_payer_summary = pd.read_sql_query(sql_query, conn)
print("--- SQL OUTPUT: Payer Billing Performance ---")
print(df_payer_summary.to_string(index=False))

--- SQL OUTPUT: Payer Billing Performance ---
insurance_provider  total_claims   total_billed  avg_claim_amount  avg_days_stay  revenue_rank
             Cigna         11249 287,139,345.16         25,525.77          15.50             1
          Medicare         11154 285,720,757.70         25,615.99          15.60             2
        Blue Cross         11059 283,254,294.21         25,613.01          15.50             3
  UnitedHealthcare         11125 282,454,542.59         25,389.17          15.50             4
             Aetna         10913 278,863,102.29         25,553.29          15.40             5


In [ ]:
import warnings
warnings.filterwarnings('ignore')

# 1. Excel Dynamic Binning
bins = [0, 10000, 25000, 40000, np.inf]
labels = ['Low ($0-$10k)', 'Moderate ($10k-$25k)', 'High ($25k-$40k)', 'Extreme (> $40k)']
df_billing['billing_tier'] = pd.cut(df_billing['billing_amount'], bins=bins, labels=labels)

# 2. Pivot Table Aggregation
excel_pivot = df_billing.pivot_table(
    index='medical_condition',
    columns='billing_tier',
    values='billing_amount',
    aggfunc=['count', 'mean'],
    fill_value=0,
    observed=False
)

# 3. Clean Currency Formatting & Render Clean Table Output
excel_pivot = excel_pivot.round(2)
print("--- EXCEL MODEL OUTPUT: Condition vs. Billing Tier Summary ---")
print(excel_pivot.to_string())

--- EXCEL MODEL OUTPUT: Condition vs. Billing Tier Summary ---
                          count                                                                 mean                                                       
billing_tier      Low ($0-$10k) Moderate ($10k-$25k) High ($25k-$40k) Extreme (> $40k) Low ($0-$10k) Moderate ($10k-$25k) High ($25k-$40k) Extreme (> $40k)
medical_condition                                                                                                                                          
Arthritis                  1745                 2807             2843             1902      5,539.97            17,511.95        32,549.08        45,202.23
Asthma                     1647                 2800             2787             1933      5,456.13            17,471.14        32,440.97        45,086.01
Cancer                     1757                 2851             2750             1850      5,530.00            17,574.35        32,397.33        45,008.40
D